In [ ]:
from pathlib import Path
import pandas as pd
import time

# Set paths.
s_path_data = Path(r"C:\2026_06_26_Hackathon\Data")
s_out_raw = s_path_data / "Police_Incidents_FULL_from_API.csv"
s_out_year_counts = s_path_data / "Police_Incidents_FULL_year_counts.csv"
s_out_crime_counts = s_path_data / "Police_Incidents_FULL_crime_type_counts.csv"

# Try the Dallas OpenData dataset ID from the Police Incidents URL.
base_url = "https://www.dallasopendata.com/resource/qv6i-rri7.csv"
limit = 50000
offset = 0
chunks = []

# Download in chunks.
while True:
    url = f"{base_url}?$limit={limit}&$offset={offset}"
    print("Downloading offset:", offset)
    df_chunk = pd.read_csv(url, dtype=str, low_memory=False)
    print("Rows downloaded:", len(df_chunk))
    if len(df_chunk) == 0:
        break
    chunks.append(df_chunk)
    offset += limit
    time.sleep(0.5)

# Combine and save.
df = pd.concat(chunks, ignore_index=True)
df.to_csv(s_out_raw, index=False)
print("TOTAL ROWS:", len(df))
print("COLUMNS:", len(df.columns))
print("Saved:", s_out_raw)

# Clean column names lightly.
df.columns = df.columns.str.strip()

# Find likely year and crime columns after API field-name conversion.
print("\nColumns:")
print(df.columns.to_list())

# Use whichever names exist.
year_col = "year_of_incident" if "year_of_incident" in df.columns else "Year of Incident"
crime_cols = [c for c in ["type_of_incident", "ucr_offense_name", "ucr_offense_description", "nibrs_crime", "nibrs_crime_category", "offense_type"] if c in df.columns]

# Save year counts.
df_year_counts = df[year_col].fillna("[NULL]").astype(str).str.strip().value_counts().reset_index()
df_year_counts.columns = [year_col, "row_count"]
df_year_counts.to_csv(s_out_year_counts, index=False)

# Save combined crime type counts.
all_counts = []
for col in crime_cols:
    df_counts = df[col].fillna("[NULL]").astype(str).str.strip().replace("", "[BLANK]").value_counts().reset_index()
    df_counts.columns = ["crime_type_value", "row_count"]
    df_counts.insert(0, "crime_type_column", col)
    all_counts.append(df_counts)

df_crime_counts = pd.concat(all_counts, ignore_index=True)
df_crime_counts.to_csv(s_out_crime_counts, index=False)

print("\nYear counts:")
print(df_year_counts.to_string(index=False))

print("\nTop crime counts:")
print(df_crime_counts.head(100).to_string(index=False))

print("\nSaved:")
print(s_out_year_counts)
print(s_out_crime_counts)

Rows downloaded: 50000
Rows downloaded: 50000
Rows downloaded: 50000
Rows downloaded: 50000
Rows downloaded: 50000
Rows downloaded: 50000
